# Fix ticker conflicts and relabel data

This notebook fixes the problem where many unrelated company names share the same wrong ticker because `yfinance.Search(...).quotes[0]` picked a bad first result.

It creates three files:

- `data/processed/ticker_conflict_report.csv`
- `data/processed/layoffs_with_tickers_cleaned.csv`
- `data/processed/labeled_merged_data_cleaned.csv`


In [1]:
from pathlib import Path
import re
from collections import Counter

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

LAYOFFS_WITH_TICKERS = DATA_DIR / "processed" / "layoffs_with_tickers.csv"
CLEANED_LAYOFFS_WITH_TICKERS = DATA_DIR / "processed" / "layoffs_with_tickers_cleaned.csv"
TICKER_CONFLICT_REPORT = DATA_DIR / "processed" / "ticker_conflict_report.csv"

MERGED_FEATURES = DATA_DIR / "processed" / "merged_bs_cs_fd.csv"
LABELED_OUTPUT = DATA_DIR / "processed" / "labeled_merged_data_cleaned.csv"

FYI_SOURCE = "fyi_layoffs.csv"

## 1. Load layoff ticker mappings

In [2]:
layoffs = pd.read_csv(LAYOFFS_WITH_TICKERS)

print("Rows:", len(layoffs))
print("Companies:", layoffs["Company_name"].nunique(dropna=True))
print("Tickers:", layoffs["Ticker"].nunique(dropna=True))
layoffs.head()

Rows: 23820
Companies: 11225
Tickers: 2695


,Company_name,layoff_size,date,source,Ticker
0,CLOUDFLARE,1100.0,2026-05-07,fyi_layoffs.csv,NET
1,BILL.COM,709.0,2026-05-07,fyi_layoffs.csv,0M5.HA
2,DEEPL,250.0,2026-05-07,fyi_layoffs.csv,DLC36116-USD
3,UPWORK,151.0,2026-05-07,fyi_layoffs.csv,UPWK
4,TRUECALLER,70.0,2026-05-07,fyi_layoffs.csv,TRUEBS.XC


## 2. Build a ticker conflict report

A ticker is suspicious when it maps to many different company names and those names do not share a common meaningful token. This catches bad mappings like `FRBT`, `VRT`, `ET`, etc.

In [3]:
GENERIC_TOKENS = {
    "A", "AN", "AND", "AT", "B", "C", "CO", "COMPANY", "COMPANIES",
    "CORP", "CORPORATION", "DBA", "D", "FKA", "FOR", "GROUP", "HOLDING",
    "HOLDINGS", "INC", "INCORPORATED", "L", "LC", "LLC", "LLP", "LP",
    "LTD", "OF", "P", "PC", "PLC", "SERVICES", "SERVICE", "THE", "TO",
}


def tokenize_company_name(name):
    text = "" if pd.isna(name) else str(name).upper()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    return [
        token
        for token in text.split()
        if len(token) >= 2 and token not in GENERIC_TOKENS
    ]


def token_set(name):
    return set(tokenize_company_name(name))


def ticker_stats(group):
    names = group["Company_name"].dropna().astype(str).drop_duplicates()
    tokenized_names = [token_set(name) for name in names]
    token_counts = Counter(token for tokens in tokenized_names for token in tokens)
    top_token, top_token_count = ("", 0)

    if token_counts:
        top_token, top_token_count = token_counts.most_common(1)[0]

    unique_companies = len(names)
    top_token_coverage = top_token_count / unique_companies if unique_companies else 0
    has_fyi = group["source"].eq(FYI_SOURCE).any()

    suspicious = unique_companies >= 10 and top_token_coverage < 0.60

    return {
        "unique_companies": unique_companies,
        "rows": len(group),
        "has_fyi": has_fyi,
        "top_token": top_token,
        "top_token_coverage": round(top_token_coverage, 3),
        "suspicious": suspicious,
    }


rows = []
for ticker, group in layoffs.dropna(subset=["Ticker"]).groupby("Ticker"):
    stats = ticker_stats(group)
    sample_names = (
        group["Company_name"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(8)
        .tolist()
    )
    rows.append({"Ticker": ticker, **stats, "sample_company_names": " | ".join(sample_names)})

report = pd.DataFrame(rows).sort_values(
    ["suspicious", "unique_companies", "rows"],
    ascending=[False, False, False],
)

report.to_csv(TICKER_CONFLICT_REPORT, index=False)

print("Suspicious tickers:", int(report["suspicious"].sum()))
report.loc[
    report["suspicious"],
    ["Ticker", "unique_companies", "rows", "top_token", "top_token_coverage", "has_fyi"],
].head(20)

Suspicious tickers: 10


,Ticker,unique_companies,rows,top_token,top_token_coverage,has_fyi
1201,FRBT,4325,7302,USA,0.033,False
2530,VRT,2595,4247,MANAGEMENT,0.030,False
1113,ET,57,73,PARTNERS,0.088,False
25,005930.KS,54,62,INTERNATIONAL,0.111,True
1094,EPD,38,49,HOTELS,0.105,False
1903,PCSGH.BK,22,28,MEDICAL,0.227,False
2314,T,21,78,GLOBAL,0.333,False
472,AHMA,21,40,MARTIN,0.095,False
2008,QCOM,13,111,QUALCOMM,0.231,True
159,0P0001X4N8,11,14,ASSOCIATES,0.273,False


## 3. Drop suspicious non-FYI mappings

The original file is not changed. This writes a new cleaned file. FYI mappings are kept because they are closer to public tech-company layoffs; suspicious WARN mappings are dropped.

In [5]:
suspicious_tickers = set(report.loc[report["suspicious"], "Ticker"])

cleaned = layoffs.copy()
cleaned["ticker_mapping_status"] = "kept"
cleaned["ticker_mapping_reason"] = "ok"

suspicious_mask = cleaned["Ticker"].isin(suspicious_tickers)
fyi_mask = cleaned["source"].eq(FYI_SOURCE)
drop_mask = suspicious_mask & ~fyi_mask

cleaned.loc[drop_mask, "ticker_mapping_status"] = "dropped"
cleaned.loc[
    drop_mask,
    "ticker_mapping_reason",
] = "ticker maps to many unrelated company names"

cleaned = cleaned.loc[~drop_mask].copy()
cleaned.to_csv(CLEANED_LAYOFFS_WITH_TICKERS, index=False)

print("Original rows:", len(layoffs))
print("Cleaned rows:", len(cleaned))
print("Dropped rows:", len(layoffs) - len(cleaned))
cleaned.head()

Original rows: 23820
Cleaned rows: 11821
Dropped rows: 11999


,Company_name,layoff_size,date,source,Ticker,ticker_mapping_status,ticker_mapping_reason
0,CLOUDFLARE,1100.0,2026-05-07,fyi_layoffs.csv,NET,kept,ok
1,BILL.COM,709.0,2026-05-07,fyi_layoffs.csv,0M5.HA,kept,ok
2,DEEPL,250.0,2026-05-07,fyi_layoffs.csv,DLC36116-USD,kept,ok
3,UPWORK,151.0,2026-05-07,fyi_layoffs.csv,UPWK,kept,ok
4,TRUECALLER,70.0,2026-05-07,fyi_layoffs.csv,TRUEBS.XC,kept,ok


## 4. Relabel merged financial data

This labels by ticker, not company name:

`merged_bs_cs_fd.company == layoffs_with_tickers_cleaned.Ticker`

In [4]:
features_df = pd.read_csv(MERGED_FEATURES)
label_layoffs = pd.read_csv(CLEANED_LAYOFFS_WITH_TICKERS)

label_layoffs["date"] = pd.to_datetime(label_layoffs["date"], errors="coerce")
label_layoffs = label_layoffs.dropna(subset=["Ticker", "date"])
label_layoffs["quarter"] = label_layoffs["date"].dt.to_period("Q").astype(str)

layoff_events = set(
    zip(
        label_layoffs["Ticker"].astype(str),
        label_layoffs["quarter"].astype(str),
    )
)

feature_quarters = pd.PeriodIndex(features_df["quarter"], freq="Q")
label_df = pd.DataFrame(
    {
        "same_quarter": feature_quarters.astype(str),
        "next_quarter": (feature_quarters + 1).astype(str),
    }
)

same_quarter_pairs = list(zip(features_df["company"].astype(str), label_df["same_quarter"]))
next_quarter_pairs = list(zip(features_df["company"].astype(str), label_df["next_quarter"]))

label_df["layoff_same_quarter"] = pd.Series(same_quarter_pairs).isin(layoff_events).astype(int).values
label_df["layoff_next_quarter"] = pd.Series(next_quarter_pairs).isin(layoff_events).astype(int).values
label_df["layoff_same_or_next_quarter"] = (
    (label_df["layoff_same_quarter"] == 1)
    | (label_df["layoff_next_quarter"] == 1)
).astype(int)

# Default target. Change this to layoff_next_quarter if you want stricter prediction.
label_df["layoff"] = label_df["layoff_same_or_next_quarter"]

dataset = pd.concat([features_df.reset_index(drop=True), label_df], axis=1)
dataset.to_csv(LABELED_OUTPUT, index=False)

print("Rows:", len(dataset))
print("Companies:", dataset["company"].nunique())
print(dataset[["layoff_same_quarter", "layoff_next_quarter", "layoff_same_or_next_quarter", "layoff"]].sum())
print("Positive companies:", dataset.loc[dataset["layoff"].eq(1), "company"].nunique())

Rows: 12047
Companies: 1941
layoff_same_quarter             712
layoff_next_quarter             656
layoff_same_or_next_quarter    1205
layoff                         1205
dtype: int64
Positive companies: 508
